In [2]:
#import libraries
import numpy as np

In [3]:
#define the environment 
environment_rows = 12
environment_columns = 12

q_values = np.zeros((environment_rows, environment_columns, 4))


In [4]:
#define actions 0 = up, 1 = right, 2 = down, 3 = left
actions = ['up', 'right', 'down', 'left']

In [5]:

rewards = np.full((environment_rows, environment_columns), -100.)
rewards[0, 3] = 10. #set the reward for the packaging area or goal to 10
rewards[0, 5] = 100. #set the reward for the packaging area  to 100


aisles = {} #store locations in a dictionary
aisles[1] = [i for i in range(1, 11)]
aisles[2] = [1, 7, 10]
aisles[3] = [i for i in range(1, 8)]
aisles[3].append(9)
aisles[4] = [3, 7]
aisles[5] = [i for i in range(11)]
aisles[6] = [5]
aisles[7] = [i for i in range(1, 10)]
aisles[8] = [3,6 ,7]
aisles[9] = [i for i in range(1,11)]
aisles[10] = [2,5]
aisles[11] = [5]



for row_index in range(1, 12):
  for column_index in aisles[row_index]:
    rewards[row_index, column_index] = -1.
  
#print rewards matrix
for row in rewards:
  print(row)

[-100. -100. -100.   10. -100.  100. -100. -100. -100. -100. -100. -100.]
[-100.   -1.   -1.   -1.   -1.   -1.   -1.   -1.   -1.   -1.   -1. -100.]
[-100.   -1. -100. -100. -100. -100. -100.   -1. -100. -100.   -1. -100.]
[-100.   -1.   -1.   -1.   -1.   -1.   -1.   -1. -100.   -1. -100. -100.]
[-100. -100. -100.   -1. -100. -100. -100.   -1. -100. -100. -100. -100.]
[  -1.   -1.   -1.   -1.   -1.   -1.   -1.   -1.   -1.   -1.   -1. -100.]
[-100. -100. -100. -100. -100.   -1. -100. -100. -100. -100. -100. -100.]
[-100.   -1.   -1.   -1.   -1.   -1.   -1.   -1.   -1.   -1. -100. -100.]
[-100. -100. -100.   -1. -100. -100.   -1.   -1. -100. -100. -100. -100.]
[-100.   -1.   -1.   -1.   -1.   -1.   -1.   -1.   -1.   -1.   -1. -100.]
[-100. -100.   -1. -100. -100.   -1. -100. -100. -100. -100. -100. -100.]
[-100. -100. -100. -100. -100.   -1.   -1. -100. -100. -100. -100. -100.]


In [6]:
#define a function if the specified location is a terminal state
def is_terminal_state(current_row_index, current_column_index):
  #if the reward = -1, then it is not a terminal state
  if rewards[current_row_index, current_column_index] == -1.:
    return False
  else:
    return True


def get_starting_location():
  #get index of a random row and column 
  current_row_index = np.random.randint(environment_rows)
  current_column_index = np.random.randint(environment_columns)

  while is_terminal_state(current_row_index, current_column_index):
    current_row_index = np.random.randint(environment_rows)
    current_column_index = np.random.randint(environment_columns)
  return current_row_index, current_column_index

#define an epsilon greedy algorithm 
def get_next_action(current_row_index, current_column_index, epsilon):
  
  if np.random.random() < epsilon:
    return np.argmax(q_values[current_row_index, current_column_index])
  else: #choose a random action
    return np.random.randint(4)


def get_next_location(current_row_index, current_column_index, action_index):
  new_row_index = current_row_index
  new_column_index = current_column_index
  if actions[action_index] == 'up' and current_row_index > 0:
    new_row_index -= 1
  elif actions[action_index] == 'right' and current_column_index < environment_columns - 1:
    new_column_index += 1
  elif actions[action_index] == 'down' and current_row_index < environment_rows - 1:
    new_row_index += 1
  elif actions[action_index] == 'left' and current_column_index > 0:
    new_column_index -= 1
  return new_row_index, new_column_index


def get_maxFRee_Space(start_row_index, start_column_index):
 
  if is_terminal_state(start_row_index, start_column_index):
    return []
  else: 
    current_row_index, current_column_index = start_row_index, start_column_index
    shortest_path = []
    shortest_path.append([current_row_index, current_column_index])
    
    while not is_terminal_state(current_row_index, current_column_index):
      
      action_index = get_next_action(current_row_index, current_column_index, 1.)
      #move to the next location 
      current_row_index, current_column_index = get_next_location(current_row_index, current_column_index, action_index)
      shortest_path.append([current_row_index, current_column_index])
    return shortest_path

In [7]:
#define training parameters
epsilon = 0.9 
discount_factor = 0.9 
learning_rate = 0.9 

for episode in range(1000):
  
  row_index, column_index = get_starting_location()

  
  while not is_terminal_state(row_index, column_index):

    action_index = get_next_action(row_index, column_index, epsilon)

    old_row_index, old_column_index = row_index, column_index 
    row_index, column_index = get_next_location(row_index, column_index, action_index)
    
    reward = rewards[row_index, column_index]
    old_q_value = q_values[old_row_index, old_column_index, action_index]
    temporal_difference = reward + (discount_factor * np.max(q_values[row_index, column_index])) - old_q_value

    # Q-value update  for the previous state as well as for the action pair
    new_q_value = old_q_value + (learning_rate * temporal_difference)
    q_values[old_row_index, old_column_index, action_index] = new_q_value

print('Training complete!')

Training complete!


In [8]:
#display output
path = get_maxFRee_Space(11, 6) 
print(path)

x= path.pop()
temp_col= x.pop()
temp_row= x.pop(0)


remain_space= rewards[temp_row,temp_col]+ rewards[11,6]

print (remain_space)


[[11, 6], [11, 5], [10, 5], [9, 5], [9, 6], [8, 6], [7, 6], [7, 5], [6, 5], [5, 5], [5, 6], [5, 7], [4, 7], [3, 7], [2, 7], [1, 7], [1, 6], [1, 5], [0, 5]]
99.0
